# [DROID-SLAM in the Wild](https://arxiv.org/pdf/2603.19076)

- Problem addressed: Traditional RGB SLAM assumes a mostly static environment, so moving objects can cause camera-tracking failures and inaccurate 3D reconstruction, especially in crowded and cluttered scenes.
- Proposed solution: DROID-W estimates per-pixel uncertainty using multi-view visual feature inconsistency and incorporates it into differentiable, uncertainty-aware Bundle Adjustment.
- Key benefit: Reduces the influence of unreliable visual observations without relying entirely on predefined dynamic-object categories, enabling more robust camera-pose estimation and 3D mapping at approximately 10 FPS.

**Problem:** Traditional SLAM assumes static scenes, so dynamic and non-rigid objects cause camera-pose errors and inaccurate 3D reconstruction.



**Limitation of existing methods:** Motion detection and segmentation methods depend on predefined dynamic-object knowledge, while existing uncertainty-aware methods often require a reliable static 3D map to estimate pixel-wise uncertainty.



**Proposed solution — DROID-W:** Extends DROID-SLAM to dynamic environments by integrating uncertainty optimization into differentiable Bundle Adjustment (BA), jointly updating pixel-wise uncertainty, camera poses, and scene geometry using multi-view visual feature similarity.



**Key advantage:** Estimates uncertainty without relying on predefined motion priors or high-quality geometric maps, improving robustness in complex, cluttered, real-world scenes.



**Additional contribution:** Introduces the DROID-W dataset with diverse outdoor dynamic scenes and YouTube videos for in-the-wild evaluation.



**Result:** Achieves state-of-the-art camera tracking and 3D geometry reconstruction in the evaluated scenarios at approximately 10 FPS.

## DROID-SLAM Section 3.1 — Preliminaries: Complete Mathematical Explanation

1. **The overall problem: what is DROID-SLAM trying to estimate?**

Imagine a camera moving through a room while capturing images.

Suppose the camera captures images

$$ \{I_t\}_{t=0}^{N} $$

Here:
- $ I_t $ is the RGB image captured at time $ t $.
- $ t $ is the frame index.
- $ N $ is the last frame index.
- The sequence contains $ N+1 $ images because indexing begins at zero.
For example,

$$ I_0,I_1,I_2,I_3,\ldots,I_N $$

As the camera moves, DROID-SLAM needs to estimate two things for each image:
- Camera pose: Where is the camera, and in which direction is it oriented?
- Scene depth: How far away are the visible scene points?
The estimates are refined repeatedly because the initial camera positions and depths may be inaccurate.

**2. Camera pose: $G_t\in SE(3)$**


The paper represents the camera pose at frame $t$ by

$$ G_t\in SE(3) $$

2.1 What is a camera pose?


A camera pose has six degrees of freedom:
- Three translations: movement along the $x$, $y$, and $z$ axes.
- Three rotations: orientation about those axes.
The pose can be represented using a rotation matrix $R_t$ and a translation vector $t_t$:

$$ G_t=
\begin{bmatrix}
R_t & t_t\\
0 & 1
\end{bmatrix} $$

where

$$ R_t\in\mathbb R^{3\times3},
\qquad
t_t\in\mathbb R^3. $$

The bottom row is $[0\;0\;0\;1]$.

The matrix is $4\times4$ because homogeneous coordinates allow rotation and translation to be combined into a single matrix multiplication.

For a 3D point $X_i$, expressed in camera $i$'s coordinate system,

$$ \begin{bmatrix}
X_j\\1
\end{bmatrix}
=
G_{ji}
\begin{bmatrix}
X_i\\1
\end{bmatrix}. $$

Equivalently,

$$ X_j=R_{ji}X_i+t_{ji}. $$

Here $G_{ji}$ is the transformation that takes a point from camera $i$'s coordinate system into camera $j$'s coordinate system.


2.2 What exactly is $SE(3)$?


$SE(3)$ stands for the Special Euclidean group in three dimensions.

It is the set of valid rigid-body transformations in 3D:

$$ SE(3)=
\left\{
\begin{bmatrix}
R&t\\
0&1
\end{bmatrix}
:
R^\top R=I,\ \det(R)=1
\right\}. $$

The conditions on $R$ mean that it represents a proper rotation: it preserves lengths and angles and does not introduce reflections.

Why is this important? A camera can move and rotate, but its rigid motion does not deform the physical scene.

# 3. Inverse depth:  $$d_t\in\mathbb R^{H/8\times W/8}$$

The paper maintains an inverse-depth map:

$$
d_t\in
\mathbb R^{\frac H8\times\frac W8}. $$

Let's unpack this notation.
- $H$: image height in pixels.
- $W$: image width in pixels.
- $H/8$ and $W/8$: height and width at one-eighth image resolution.
- Each entry in $d_t$ represents the inverse depth of a pixel at that lower resolution.



For an original image of $640\times480$, for example, the corresponding map would be $80\times60$, assuming the dimensions are divisible by eight.

**Why is the map at one-eighth resolution?**


Dense optimization over every full-resolution pixel would be expensive.


DROID-SLAM operates on a lower-resolution grid for its depth and correspondence-related representations, reducing the number of variables.

If the image is downsampled by eight in both dimensions, the number of grid positions becomes approximately

$$ \frac H8\frac W8=\frac{HW}{64}. $$

That is one sixty-fourth as many positions as in the full-resolution image.

This does not mean the original RGB image must itself be only one-eighth resolution; it describes the spatial resolution of the particular state variables and grids in the paper.

##The frame graph $(V,E)d$


The paper constructs a frame graph:
$$
\mathcal G=(V,E).
$$

Here, the symbol $\mathcal Gd$ denotes a graph; it is different from the pose matrix $G_td$.
- $Vd$: the set of image frames.
- $Ed$: the set of selected pairs of frames that are connected for optimization.
For example, suppose the camera captures four images:
$$
I_0,\ I_1,\ I_2,\ I_3.
$$

The graph might contain the edges
$$
E=\{(0,1),(1,2),(2,3),(0,2)\}.
$$


Adjacent frames are connected; other overlapping pairs may also be connected.


An edge $(i,j)\in Ed$ means that the two images have sufficient co-visibility: they observe some of the same scene.

For instance, both images may contain the same wall corner, table edge, or chair.

Why use a graph instead of comparing every image with every other image?

Not all image pairs are equally useful. Two images captured far apart may have little overlap. A graph lets the system select useful frame pairs for tracking and optimization.

For each selected pair $(i,j)d$, DROID-SLAM tries to make the geometry predicted from the estimated camera poses and depths agree with the image-to-image correspondences.

**5. Differential bundle adjustment (BA)**


Bundle adjustment is an optimization process that jointly refines:
1. Camera poses.
2. Scene geometry, represented here by inverse depths.


The word differentiable means that the geometric operations can be differentiated with respect to the quantities being optimized.

Their derivatives tell the optimizer how changing a pose or depth would change the predicted pixel locations.

DROID-SLAM uses a differentiable BA layer so that the geometric optimization can also participate in learning the neural network.


# 6. Equation (1): predicting where a pixel should appear in another image


The paper writes

$$ \boxed{
p_{ij}=
\Pi_c\left(
G'_{ij}\circ
\Pi_c^{-1}(p_i,d'_i)
\right)
}
\tag{1} $$

Read this from the inside outward.
1. Start with a pixel in image $i$.
2. Use its inverse depth to reconstruct a 3D point.
3. Transform that point into the coordinate system of camera $j$.
4. Project the transformed point into image $j$.
5. Obtain the predicted pixel correspondence $p_{ij}$.
The primes indicate current estimates or optimization variables in the paper's notation.

Let's derive each operation.
6.1 The pixel coordinate $p_i$
The paper defines

$$ p_i\in
\mathbb R^{\frac H8\times\frac W8\times2}. $$

At each position in the lower-resolution image grid, the coordinate has two components:

$$ p_i=
\begin{bmatrix}
u_i\\v_i
\end{bmatrix}. $$

Here:
- $u_i$: horizontal pixel coordinate.
- $v_i$: vertical pixel coordinate.
For example, one grid position might be

$$ p_i=
\begin{bmatrix}
40\\30
\end{bmatrix}. $$

This identifies a pixel location in image $i$. It is not yet a 3D point.

**6.2 What does $\Pi_c^{-1}(p_i,d'_i)d$ mean?**


$\Pi_c^{-1}d$ is the inverse camera projection operation. It converts a pixel coordinate and its estimated inverse depth into a 3D point in the camera's coordinate system.

First, recover depth from inverse depth:

$$ z_i=\frac1{d'_i}.$$

Next, use the camera intrinsics.

Let

$$ K=
\begin{bmatrix}
f_x&0&c_x\\
0&f_y&c_y\\
0&0&1
\end{bmatrix},$$

where:
- $f_x,f_yd$: focal lengths measured in pixel units.
- $c_x,c_yd$: principal point coordinates.
- $Kd$: intrinsic calibration matrix.
For a pinhole camera, the projection equations are

$$ u=f_x\frac XZ+c_x,
\qquad
v=f_y\frac YZ+c_y.$$

To reverse them, solve for $Xd$ and $Yd$:

$$ X=\frac{u-c_x}{f_x}Z,
\qquad
Y=\frac{v-c_y}{f_y}Z.$$

Substituting $Z=z_i=1/d'_id$,

$$ \boxed{
X_i=
\begin{bmatrix}
\frac{u_i-c_x}{f_xd'_i}\\
\frac{v_i-c_y}{f_yd'_i}\\
\frac1{d'_i}
\end{bmatrix}
}$$

This is the back-projected 3D point.

Important: A pixel alone does not specify a unique 3D point. It specifies a ray extending out from the camera. The depth tells us where along that ray the point lies.

**6.3 A numerical example of back-projection**


Suppose the camera parameters are

$$ f_x=f_y=100,\qquad c_x=c_y=50. $$

Consider the pixel

$$ p_i=(60,70) $$

with estimated inverse depth

$$ d'_i=0.5\ \text{m}^{-1}. $$

First, compute depth:

$$ z_i=\frac1{0.5}=2\text{ m}. $$

Next, compute the 3D coordinates:

$$ X=\frac{60-50}{100}(2)=0.2\text{ m}, $$


$$ Y=\frac{70-50}{100}(2)=0.4\text{ m}, $$


$$ Z=2\text{ m}. $$

Therefore,

$$ \boxed{X_i=(0.2,\ 0.4,\ 2)\text{ m}} $$

The original pixel has now been converted into a 3D point expressed in camera $i$'s coordinate system.

# 6.4 What does $G'_{ij}$ mean?


$G'_{ij}$ is the estimated relative camera pose that transforms points from camera $i$'s coordinates into camera $j$'s coordinates.

Write it as

$$ G'_{ij}=
\begin{bmatrix}
R'_{ij}&t'_{ij}\\
0&1
\end{bmatrix}. $$

The 3D transformation is

$$ \boxed{
X_j=R'_{ij}X_i+t'_{ij}
} $$

This transformation accounts for both rotation and translation.

For example, suppose the second camera is translated one metre to the right relative to the first camera, with no rotation. Under this particular transformation convention,

$$ R'_{ij}=I,\qquad
t'_{ij}=
\begin{bmatrix}
-1\\0\\0
\end{bmatrix}. $$

For the point calculated above,

$$ X_i=
\begin{bmatrix}
0.2\\0.4\\2
\end{bmatrix}, $$

we obtain

$$ X_j=
\begin{bmatrix}
0.2\\0.4\\2
\end{bmatrix}
+
\begin{bmatrix}
-1\\0\\0
\end{bmatrix}
=
\boxed{
\begin{bmatrix}
-0.8\\0.4\\2
\end{bmatrix}
}. $$

The point has not physically moved. Only its coordinates relative to the camera have changed.

The sign of the translation depends on the chosen pose convention; the important requirement is that the transformation consistently maps coordinates from frame $i$ to frame $j$.

# 6.5 What does $\Pi_c(X_j)$ do?
The projection operation maps a 3D point into a 2D pixel:

$$ \Pi_c(X_j)=
\begin{bmatrix}
f_xX_j/Z_j+c_x\\
f_yY_j/Z_j+c_y
\end{bmatrix}. $$

Using the numerical example,

$$ X_j=(-0.8,0.4,2), $$

we obtain

$$ u_j=100\frac{-0.8}{2}+50=10, $$


$$ v_j=100\frac{0.4}{2}+50=70. $$

Therefore,

$$ \boxed{p_{ij}=(10,70)} $$

This is the pixel location predicted in image $j$ for the 3D point reconstructed from image $i$.

**6.6 Put the entire Equation (1) together**

$$ \underbrace{p_i}_{\text{pixel in image }i}
\xrightarrow{\Pi_c^{-1}(\cdot,d'_i)}
\underbrace{X_i}_{\text{3D point}}
\xrightarrow{G'_{ij}}
\underbrace{X_j}_{\text{3D point in camera }j}
\xrightarrow{\Pi_c}
\underbrace{p_{ij}}_{\text{predicted pixel in image }j}. $$

So Equation (1) is simply a back-project → transform → project operation.

The mathematical purpose is to predict where each scene point should appear in another frame if the current pose and depth estimates are correct.

# 7. Predicted correspondence versus neural-network correspondence


This is one of the most important distinctions in DROID-SLAM.

The paper uses two different quantities:

$$ p_{ij}
\qquad\text{and}\qquad
p^*_{ij}. $$

They are not the same.
7.1 Geometric prediction: \(p_{ij}\)
As derived above,

$$ p_{ij}=
\Pi_c\left(
G'_{ij}\circ\Pi_c^{-1}(p_i,d'_i)
\right). $$

This is computed from the current pose and inverse-depth estimates.

It answers:
If my current camera poses and depths are correct, where should this pixel appear in the other image?

7.2 Neural-network prediction: \(p^*_{ij}\)
DROID-SLAM's learned correspondence system estimates a dense two-dimensional correspondence field:

$$ p^*_{ij}\in
\mathbb R^{\frac H8\times\frac W8\times2}. $$

For each grid position in image \(i\), the network estimates the corresponding location in image \(j\).

For example, suppose the pixel at

$$ p_i=(60,70) $$

has a network-predicted correspondence

$$ p^*_{ij}=(12,68). $$

This is the correspondence estimate supplied by the neural network, not the projection computed using the current geometry.

The two values answer different questions:
Quantity	How it is obtained	Meaning
\(p_{ij}\)	Camera geometry and current depth	Where the point should appear according to the current 3D model
\(p^*_{ij}\)	Learned correspondence system	Where the network estimates the point appears in the other image
The optimization adjusts the camera poses and depths to make these two estimates agree.
7.3 What if the two disagree?
Suppose

$$ p_{ij}=(10,70) $$

but

$$ p^*_{ij}=(12,68). $$

Then the discrepancy is

$$ r_{ij}=p^*_{ij}-p_{ij} $$


$$ r_{ij}=
\begin{bmatrix}
12\\68
\end{bmatrix}
-
\begin{bmatrix}
10\\70
\end{bmatrix}
=
\begin{bmatrix}
2\\-2
\end{bmatrix}. $$

This means the geometrically predicted pixel is two pixels too far left and two pixels too far down relative to the network's estimated correspondence.

The Euclidean error is

$$ \|r_{ij}\|_2
=\sqrt{2^2+(-2)^2}
=\sqrt8
\approx2.83\text{ pixels}. $$

The optimizer can reduce this error by adjusting the pose, the depth, or both.

One correspondence does not generally tell us uniquely which variable is wrong. Multiple pixels and frame pairs help constrain the joint solution.

**8. Confidence map $w_{ij}$**
The paper also states that DROID-SLAM predicts a confidence map:

$$w_{ij}\in
\mathbb R^{\frac H8\times\frac W8\times2}
$$

using the dimensions written in the supplied text.

The purpose of this map is to determine how strongly each correspondence residual should influence the optimization.

Why is that necessary?

Not every correspondence is equally reliable. A featureless wall, an occluded region, a moving person, or a repetitive texture may produce an uncertain correspondence.

If every pixel were trusted equally, incorrect correspondences could distort the estimated camera motion and scene depth.
8.1 What does the confidence do?
Conceptually:
- High confidence means the correspondence is trusted more.
- Low confidence means the correspondence is trusted less.
The exact interpretation of $w_{ij}$ in the objective is important: it is used as a weight or, equivalently under the usual Gaussian-error interpretation, as an inverse-uncertainty quantity.

We will now see how the equation implements this weighting.

**9. The bundle-adjustment objective**
The paper writes the energy as

$$ \boxed{
E(G',d')=
\sum_{(i,j)\in E}
\|p^*_{ij}-p_{ij}\|^2_{\Sigma_{ij}}
} $$

where

$$ \boxed{\Sigma_{ij}=\operatorname{diag}(w_{ij})}. $$

This equation is the central objective of the section.

Let's unpack every symbol.

**9.1 What are $G'$ and $d'$?**
These represent the collection of pose and inverse-depth estimates being optimized.

Conceptually,

$$ G'=\{G'_0,G'_1,\ldots,G'_N\} $$

and

$$ d'=\{d'_0,d'_1,\ldots,d'_N\}. $$

The exact pose parameterization and gauge convention are implementation details; these collections represent the variables of the optimization.

**9.2 What does the summation mean?**

$$ \sum_{(i,j)\in E} $$

means that the objective accumulates correspondence errors across the selected image pairs.

For every edge $ (i,j) $:
1. Reconstruct scene points using image $ i $'s inverse depth.
2. Transform and project them into image $ j $.
3. Compare the projected locations with the network's correspondence estimates.
4. Weight the discrepancies using the confidence information.
5. Add the weighted errors to the total objective.


The optimizer attempts to minimize this total error, not just the error in one image pair.

**9.3 Define the residual**
Let

$$ r_{ij}=p^*_{ij}-p_{ij}. $$

This is called the residual.

For each pixel, the residual has two components:

$$ r_{ij}=
\begin{bmatrix}
r_u\\r_v
\end{bmatrix}. $$

These are the horizontal and vertical correspondence errors.

If the predicted and network-estimated correspondences agree perfectly,

$$ p^*_{ij}=p_{ij} $$

and therefore

$$ r_{ij}=
\begin{bmatrix}
0\\0
\end{bmatrix}. $$

The residual contribution is then zero.

**9.4 What is a Mahalanobis distance?**


The paper uses the notation

$$ \|r\|^2_{\Sigma}. $$

The conventional definition of a squared Mahalanobis norm is

$$ \boxed{
\|r\|^2_{\Sigma}=r^\top\Sigma^{-1}r
} $$

when $\Sigma$ is a covariance matrix.

However, the paper explicitly defines

$$ \Sigma_{ij}=\operatorname{diag}(w_{ij}) $$

and describes $w_{ij}$ as a confidence map. In that case, the intended weighted-residual interpretation is typically

$$ \boxed{
E_{ij}=r_{ij}^\top W_{ij}r_{ij}
} $$

where $W_{ij}$ is the diagonal matrix of correspondence weights.

# 10. What exactly is being minimized?

Putting the preceding equations together, the intended optimization is conceptually

$$ \boxed{
\min_{G',d'}
\sum_{(i,j)\in E}
\sum_{p}
r_{ij}(p)^\top
W_{ij}(p)
r_{ij}(p)
} $$

where $ p $ runs over the pixel grid for each selected image pair.

Recall that

$$ r_{ij}(p)
=
p^*_{ij}(p)
-
\Pi_c\left(
G'_{ij}\circ
\Pi_c^{-1}(p,d'_i(p))
\right). $$

This gives the complete dependency:

$$ \boxed{
\text{Pose and depth}
\longrightarrow
\text{projected pixels}
\longrightarrow
\text{residuals}
\longrightarrow
\text{weighted error}.
} $$

The camera poses and inverse depths are the quantities the optimizer changes.

The learned correspondences and their confidence values provide the targets and weights for that optimization step.


**Why are pose and depth optimized jointly?**
Suppose the predicted pixel is incorrect.
- Perhaps the camera translation is wrong.
- Perhaps the camera rotation is wrong.
- Perhaps the estimated scene depth is wrong.
- Perhaps several of these variables are wrong simultaneously.
A change in camera pose affects the projections of many scene points. A change in one pixel's inverse depth affects the projection of that particular point into connected frames.

Joint optimization allows the system to reconcile all these constraints together.

# Gauss–Newton optimization: why do we need it?

DROID-SLAM uses the Gauss–Newton algorithm to obtain an iterative solution.

The fundamental idea is:
1. Start with the current pose and depth estimates.
2. Calculate the current residuals.
3. Approximate how the residuals change for small parameter updates.
4. Solve a linear system for those updates.
5. Apply the updates.
6. Recompute the residuals and repeat.
Let's derive the mathematics behind these steps.

Linearizing the residual
Let the optimization variables collectively be

$$ x=
\begin{bmatrix}
\xi\\d
\end{bmatrix}. $$

Here:
- $\xi$ represents the pose parameters.
- $d$ represents the inverse-depth parameters.
The pose parameters are not simply the nine entries of a rotation matrix plus three translations.

A rigid pose has six local degrees of freedom, so its update is usually represented by a six-dimensional increment.

For a small update $\Delta x$, the first-order Taylor approximation of a residual is

$$ \boxed{
r(x+\Delta x)
\approx
r(x)+J\Delta x
} $$

where

$$ \boxed{
J=\frac{\partial r}{\partial x}
} $$

is the Jacobian matrix.


**12.1 What does the Jacobian tell us?**


The Jacobian measures how sensitive the residual is to each optimization variable.

For example:

$$ \frac{\partial r}{\partial \xi} $$

describes how the correspondence error changes when the camera pose changes.

Similarly,

$$ \frac{\partial r}{\partial d} $$

describes how the correspondence error changes when inverse depth changes.

For the projection model, depth influences the 3D coordinates and hence the projected pixel locations. Camera pose influences the transformation of the 3D points. The Jacobian captures both effects.


**12.2 Why is the approximation useful?**

The original relationship between pose, depth, and pixel coordinates is nonlinear. For small changes, the Taylor approximation makes it locally linear.

This allows the optimizer to calculate a useful update using linear algebra.

After applying the update, the system recomputes the geometry and linearizes again. This is why Gauss–Newton is an iterative method.

**Deriving the Gauss–Newton normal equations**

Consider the weighted least-squares objective for the residual vector $r$:

$$ E(x)=r(x)^\top Wr(x), $$

where $W$ is a positive semidefinite weight matrix.

Using the linearized residual,

$$ r(x+\Delta x)\approx r+J\Delta x, $$

we approximate the objective after the update:

$$ E(\Delta x)
\approx
(r+J\Delta x)^\top W(r+J\Delta x). $$

Expand the expression:

$$ \begin{aligned}
E(\Delta x)\approx {}&
r^\top Wr\\
&+r^\top WJ\Delta x\\
&+\Delta x^\top J^\top Wr\\
&+\Delta x^\top J^\top WJ\Delta x.
\end{aligned} $$

For symmetric $W$, the two middle terms are equal scalars. Therefore,

$$ E(\Delta x)\approx
r^\top Wr
+2\Delta x^\top J^\top Wr
+\Delta x^\top J^\top WJ\Delta x. $$

Differentiate with respect to $\Delta x$:

$$ \frac{\partial E}{\partial\Delta x}
=
2J^\top Wr+2J^\top WJ\Delta x. $$

At the minimum of this local quadratic approximation, set the derivative to zero:

$$ 2J^\top Wr+2J^\top WJ\Delta x=0. $$

Divide by two:

$$ J^\top WJ\Delta x=-J^\top Wr. $$

Define

$$ H=J^\top WJ $$

and

$$ b=-J^\top Wr. $$

Then the normal equations become

$$ \boxed{H\Delta x=b}. $$

This is the basic Gauss–Newton update equation.

The matrix $H$ approximates the Hessian (the matrix of second derivatives) of the objective, while $b$ is the negative gradient.

Important: The Gauss–Newton approximation neglects second-derivative terms involving the residual itself. It is not generally the exact Hessian of the nonlinear objective.

**why the system has a block-matrix structure**


The paper writes

$$ \boxed{
\begin{bmatrix}
B&E\\
E^\top&C
\end{bmatrix}
\begin{bmatrix}
\Delta\xi\\
\Delta d
\end{bmatrix}
=
\begin{bmatrix}
v\\
w
\end{bmatrix}
}
\tag{2} $$

This is the Gauss–Newton normal equation reorganized into pose variables and inverse-depth variables.

The symbol $E$ here is a matrix of pose–depth coupling terms. It is not the frame-edge set $E$ introduced earlier; the paper reuses the same letter for two different things.

Let's examine every block.


**The pose update:$\Delta\xi$**


$$ \Delta\xi $$

represents the collection of small pose increments.

If there are $N+1$ frames and six local pose parameters per frame, the full pose increment would nominally have $6(N+1)$ entries before accounting for any fixed reference pose.

For a particular camera, the increment can be represented as

$$ \Delta\xi_i=
\begin{bmatrix}
\Delta t_x\\
\Delta t_y\\
\Delta t_z\\
\Delta\theta_x\\
\Delta\theta_y\\
\Delta\theta_z
\end{bmatrix}, $$

where the first three entries are translation increments and the last three are local rotation increments.

The precise ordering and left- or right-multiplication convention depend on the implementation.

The updated pose is obtained using a pose-manifold update, schematically,


$$ G_i^{\text{new}}
=
\operatorname{Exp}(\Delta\xi_i^\wedge)G_i^{\text{old}}, $$

for a left-multiplicative convention.

Here, $\operatorname{Exp}$ maps the local rigid-motion increment into an $SE(3)$ transformation, and the hat operator $(\cdot)^\wedge$ converts a six-vector into the corresponding Lie-algebra matrix.

The exact multiplication order is convention-dependent, but the principle is that pose updates preserve the rigid-transform structure.

**The inverse-depth update: $\Delta d$**

$$ \Delta d $$

contains the changes to the inverse-depth estimates.

For example, if a particular inverse-depth value is

$$ d_i=0.5, $$

and its update is

$$ \Delta d_i=0.02, $$

then a simple additive update gives

$$ d_i^{\text{new}}=0.5+0.02=0.52. $$

Its corresponding depth changes from

$$ z_i^{\text{old}}=\frac1{0.5}=2 $$

to

$$ z_i^{\text{new}}=\frac1{0.52}\approx1.923. $$

Thus, a small change in inverse depth changes the reconstructed 3D point, which changes its projection and therefore the correspondence residual.


**The matrix $B$: pose–pose interactions**

$$ B $$

is the pose-related block of the Gauss–Newton system.

It captures how the objective changes when camera poses change.

For example, the relative position of two cameras affects the projections of points observed in both images.

Consequently, changing one camera's pose can change residuals associated with many correspondences.

The pose block accumulates the relevant pose Jacobian products:

$$ B\sim J_\xi^\top WJ_\xi. $$

The symbol $\sim$ here indicates the conceptual relationship between the block and the corresponding Jacobian contribution; exact accumulation depends on the system's residual organization.


**The matrix $C$: depth–depth interactions**

$$ C $$

is the inverse-depth-related block.

Conceptually,

$$ C\sim J_d^\top WJ_d. $$

It captures how changes in the inverse-depth variables affect the residuals.

The excerpt specifically says that $C$ is diagonal because each term in Equation (2) depends only on an individual inverse-depth variable within the depth block.

This sparsity is crucial: it means the depth-related linear system is especially cheap to solve.

If

$$ C=
\begin{bmatrix}
c_1&0&0\\
0&c_2&0\\
0&0&c_3
\end{bmatrix}, $$

then

$$ C^{-1}=
\begin{bmatrix}
1/c_1&0&0\\
0&1/c_2&0\\
0&0&1/c_3
\end{bmatrix}, $$

provided all diagonal entries are nonzero.

Instead of solving a dense linear system, we can divide each component by its corresponding diagonal entry.

In practical optimization, singular or poorly conditioned variables may require damping, regularization, or special handling.


**The matrix $E$: pose–depth coupling
The off-diagonal**

$$ E $$

captures the interaction between pose updates and inverse-depth updates.

Conceptually,

$$ E\sim J_\xi^\top WJ_d. $$

Why is there a coupling?

Because the projected pixel depends on both pose and depth.

Suppose a point appears at the wrong location in image $j$. Changing the camera's translation could move its projection.

Changing the point's inverse depth could also move its projection under camera translation.

The optimizer must account for the fact that these variables can compensate for one another.

The transpose $E^\top$ appears in the lower-left block because the Gauss–Newton normal matrix is symmetric under the usual least-squares construction.


**The vectors $v$ and $w$**


The right-hand side is

$$ \begin{bmatrix}
v\\w
\end{bmatrix}. $$

These are the gradient-related terms of the linear system:
- $v$: pose-related update target.
- $w$: inverse-depth-related update target.
They are assembled from the current residuals and their Jacobians, analogous to

$$ v\sim-J_\xi^\top Wr,
\qquad
w\sim-J_d^\top Wr. $$

The exact signs depend on whether the residual is defined as observed minus predicted or predicted minus observed, and on the paper's normal-equation convention.

The system solves for $\Delta\xi$ and $\Delta d$ together, taking account of their coupling.

**Deriving the Schur-complement solution step by step**


The paper writes

$$ \boxed{
\Delta\xi=
[B-EC^{-1}E^\top]^{-1}
(v-EC^{-1}w)
} $$


and

$$ \boxed{
\Delta d=C^{-1}(w-E^\top\Delta\xi)
}
\tag{3}
 $$

These equations are derived directly from Equation (2).
No new optimization objective is introduced.

The purpose is to solve for the pose update and depth update efficiently by exploiting the structure of the matrix.

Let's derive them from scratch.

Start with

$$ \begin{bmatrix}
B&E\\
E^\top&C
\end{bmatrix}
\begin{bmatrix}
\Delta\xi\\
\Delta d
\end{bmatrix}
=
\begin{bmatrix}
v\\w
\end{bmatrix}. $$

Matrix multiplication gives two equations.

The first row is

$$ \boxed{
B\Delta\xi+E\Delta d=v
}
\tag{4} $$

The second row is

$$ \boxed{
E^\top\Delta\xi+C\Delta d=w
}
\tag{5} $$

These equations are coupled: both contain pose and depth updates.

Our strategy is to solve Equation (5) for the depth update, substitute it into Equation (4), and then solve for the pose update.

$\Delta dd$
Start from Equation (5):

$$ E^\top\Delta\xi+C\Delta d=w. $$

Move the first term to the right-hand side:

$$ C\Delta d=w-E^\top\Delta\xi. $$

Multiply both sides by $C^{-1}d$:

$$ C^{-1}C\Delta d
=
C^{-1}(w-E^\top\Delta\xi). $$

Since

$$ C^{-1}C=I, $$

we obtain

$$ \boxed{
\Delta d=C^{-1}(w-E^\top\Delta\xi)
}
\tag{6} $$

This is already the second equation in Equation (3).

But we still need to find $\Delta\xi$.

Notice that Equation (6) expresses the depth update in terms of the pose update. It does not yet independently determine either one.


Recall Equation (4):

$$ B\Delta\xi+E\Delta d=v. $$

Substitute Equation (6):

$$ B\Delta\xi+
E\left[C^{-1}(w-E^\top\Delta\xi)\right]
=v. $$

Distribute the matrix multiplication:

$$ B\Delta\xi+
EC^{-1}w
-
EC^{-1}E^\top\Delta\xi
=v. $$

Now collect all terms containing $\Delta\xi$:

$$ B\Delta\xi
-
EC^{-1}E^\top\Delta\xi
=
v-EC^{-1}w. $$

Factor out $\Delta\xi$ on the left:

$$ \boxed{
(B-EC^{-1}E^\top)\Delta\xi
=
v-EC^{-1}w
}
\tag{7} $$

This equation contains only the pose increment as an unknown.

From Equation (7),

$$ (B-EC^{-1}E^\top)\Delta\xi
=
v-EC^{-1}w. $$

Multiply both sides by the inverse of the left-hand matrix:

$$ (B-EC^{-1}E^\top)^{-1}
(B-EC^{-1}E^\top)\Delta\xi $$


$$ =
(B-EC^{-1}E^\top)^{-1}
(v-EC^{-1}w). $$

The matrix and its inverse cancel, giving

$$ \boxed{
\Delta\xi=
(B-EC^{-1}E^\top)^{-1}
(v-EC^{-1}w)
} $$

This is the first equation in Equation (3).

The matrix

$$ \boxed{S=B-EC^{-1}E^\top} $$

is called the Schur complement of $Cd$ in the block system.

In actual numerical implementations, it is usually preferable to solve a linear system involving $Sd$ rather than explicitly calculate its matrix inverse. The inverse notation in the paper expresses the mathematical solution.


Once the pose update $\Delta\xi$ is known, substitute it into Equation (6):

$$ \boxed{
\Delta d=C^{-1}(w-E^\top\Delta\xi)
} $$

Now the depth update can be computed.

Because $Cd$ is diagonal according to the paper's stated formulation, applying $C^{-1}d$ is inexpensive: each component is scaled by the inverse of its corresponding diagonal entry.

**Why does the Schur complement make bundle adjustment efficient?**


Suppose a scene contains thousands of pixel locations across multiple images.

There may be:
- Many camera pose parameters.
- A very large number of inverse-depth variables.
- Coupling between camera poses and depth estimates.
Solving one giant dense system directly can be expensive.

The Schur-complement method exploits the structure of the depth block $C$.

Because $C$ is diagonal in the formulation described in the excerpt:

$$ C^{-1} $$

is inexpensive to apply.

The system can therefore eliminate depth updates and solve the reduced pose system:

$$ \boxed{
(B-EC^{-1}E^\top)\Delta\xi
=
v-EC^{-1}w
} $$

Then the depth updates are recovered using

$$ \boxed{
\Delta d=C^{-1}(w-E^\top\Delta\xi)
}. $$

The reduced pose system can still be large, and the Schur complement is not automatically diagonal. Its efficiency comes from exploiting the structure of the depth variables and using appropriate sparse linear algebra.

A useful distinction is that eliminating depth variables does not mean ignoring depth. Their influence is incorporated into the reduced pose system through the term $EC^{-1}E^\top$.

# Uncertainty-aware Bundle Adjustment (UBA): Full step-by-step mathematical derivation


The main idea of this section is to reduce the influence of moving objects on camera-pose and depth estimation in DROID-SLAM.

For example, imagine a camera observing a room containing a table and a person walking across it. The table is approximately static, so its image motion is consistent with camera motion. The person is moving independently, so their image motion cannot be explained entirely by the camera's motion.

Ordinary bundle adjustment may try to explain both motions using the same camera pose and scene geometry. Uncertainty-aware bundle adjustment reduces the influence of unreliable correspondences.

The paper uses maps of size $H/8\times W/8$. Thus, the uncertainty is predicted at one-eighth the input image height and width, rather than independently at every full-resolution pixel.

**Important distinction**:
- the paper's uncertainty $u_i'$ is a learned or optimized uncertainty quantity, while the pose and depth are geometric state variables.  

- The exact interpretation of $u_i'$ as a variance, inverse variance, or a transformed uncertainty depends on the paper's preceding definitions and implementation.
- We can derive the displayed equations without assuming a parameterization that is not shown in your excerpt.


# Start with ordinary bundle adjustment

Bundle adjustment (BA) estimates camera poses and scene geometry by minimizing the disagreement between observed image correspondences and geometrically predicted correspondences.

Suppose frame $i$ contains a pixel $p_i$.
Geometric projection pipeline
Pixel $p_i$ + inverse depth $d_i(p_i)$


Recover the 3D point in camera $i$


Transform the point from camera $i$ to camera $j$


Project into frame $j$ to obtain $p_{ij}$


The predicted correspondence $p_{ij}$ depends on the camera poses and depth.

Let $K$ be the camera intrinsic matrix and $\tilde p_i=(x_i,y_i,1)^T$ be a homogeneous pixel coordinate. With inverse depth $d_i(p_i)$, the 3D point in camera $i$ is

$$ X_i=\frac{1}{d_i(p_i)}K^{-1}\tilde p_i. $$

Transforming it into camera $j$'s coordinate system gives

$$ X_j=G_jG_i^{-1}X_i, $$

assuming $G_i,G_j$ map camera coordinates into a common world coordinate system.

The projected pixel is

$$ p_{ij}=\pi(KX_j), $$

where $\pi$ performs perspective division:

$$ \pi\left(
\begin{bmatrix}X\\Y\\Z\end{bmatrix}
\right)
=
\begin{bmatrix}X/Z\\Y/Z\end{bmatrix}. $$

Therefore, the predicted correspondence can be written compactly as

$$ \boxed{p_{ij}=\Pi(G_i,G_j,d_i,p_i)} $$

where $\Pi$ represents the entire unprojection-transform-projection process.

In DROID-SLAM, differentiable geometric projection lets the system compute how this correspondence changes when poses and depth change.

# Derive the ordinary reprojection residual


Suppose the network estimates the target location $p_{ij}^{*}$, while geometry predicts $p_{ij}$.

Their difference is the reprojection residual:

$$ \boxed{r_{ij}=p_{ij}^{*}-p_{ij}} $$

For a two-dimensional pixel, this is a vector:

$$ r_{ij}=
\begin{bmatrix}
r_x\\
r_y
\end{bmatrix}
=
\begin{bmatrix}
x_{ij}^{*}-x_{ij}\\
y_{ij}^{*}-y_{ij}
\end{bmatrix}. $$

The ordinary squared reprojection error is

$$ \|r_{ij}\|^2=r_{ij}^{T}r_{ij}
=r_x^2+r_y^2. $$

Across all connected frame pairs and pixels, ordinary BA minimizes an aggregate error of the form

$$ E_{\mathrm{BA}}(G,d)
=
\sum_{(i,j)\in E}
\|p_{ij}^{*}-p_{ij}\|^2. $$

This objective treats large residuals as costly. That is often desirable for static scene points, but problematic for independently moving objects.

# Why dynamic objects create a problem


Consider two pixels with the same residual magnitude:
- Static table pixel: residual \(r=2\) pixels.
- Moving-person pixel: residual \(r=10\) pixels.


Ordinary squared errors are

$$ E_{\mathrm{table}}=2^2=4 $$


$$ E_{\mathrm{person}}=10^2=100. $$

The moving person's residual contributes 25 times as much error as the table pixel's residual.

The optimizer may consequently try to change the camera pose or scene depth to explain the person's independent motion. This can damage the estimates for the static scene.

The solution is to introduce a pixel-dependent uncertainty that determines how strongly each residual influences optimization.

# Deriving the uncertainty-aware weight


The paper defines

$$ \boxed{
\Sigma_{ij}^{\mathrm{uncer}}
=
\operatorname{diag}\left(
w_{ij}\cdot\frac{1}{u_i'}
\right)
}
\tag{4} $$

Let's break this down mathematically.

**Step 5.1 — Begin with the existing weight**

Let $w_{ij}$ represent the existing correspondence weight. It captures the baseline weighting or confidence assigned to a correspondence.

A larger $w_{ij}$ means that the correspondence receives more influence, all else being equal.

**Step 5.2 — Introduce uncertainty**

Let $u_i'>0$ be the uncertainty associated with the pixel.

The paper uses its reciprocal:

$$ q_i=\frac{1}{u_i'}. $$

This reciprocal makes the effective influence decrease as uncertainty increases:

$$ u_i'\uparrow
\quad\Longrightarrow\quad
\frac{1}{u_i'}\downarrow. $$

Conversely,

$$ u_i'\downarrow
\quad\Longrightarrow\quad
\frac{1}{u_i'}\uparrow. $$

Thus, larger uncertainty means less weight, while smaller uncertainty means more weight.

**Step 5.3 — Combine the existing weight and uncertainty**

Multiplying the two terms gives the effective scalar weight:

$$ \boxed{
q_{ij}=\frac{w_{ij}}{u_i'}
} $$

This is the central idea of Equation (4): combine the original correspondence weight with a new uncertainty-dependent factor.

For a residual vector with two components, we can illustrate the weighting matrix as

$$ \Sigma_{ij}^{\mathrm{uncer}}
=
\begin{bmatrix}
w_{ij}/u_i'&0\\
0&w_{ij}/u_i'
\end{bmatrix}. $$

Here, we assume the same scalar weight applies to both pixel-coordinate components.

The exact matrix dimensions depend on how the paper implements its correspondence residuals.

**Step 5.4 — Why use $\operatorname{diag}(\cdot)$?**


The notation $\operatorname{diag}$ constructs a diagonal matrix from the specified weights.

For example,

$$ \operatorname{diag}(a,b)=
\begin{bmatrix}
a&0\\
0&b
\end{bmatrix}. $$

Diagonal weighting means that each residual component is weighted without introducing off-diagonal coupling between components.

For a dense collection of pixel correspondences, the implementation can use a diagonal weight for each pixel or correspondence rather than constructing a large dense matrix explicitly.

**Interpretation of Equation (4):**

the uncertainty is converted into a confidence-like weight and incorporated into the optimization.

This gives reliable, low-uncertainty pixels more influence and unreliable, high-uncertainty pixels less influence.

# Derive the uncertainty-aware Mahalanobis distance


The paper then uses the uncertainty-aware weighted norm in Equation (5).

The squared Mahalanobis norm associated with a weighting or information matrix $W$ is

$$ \boxed{\|r\|_W^2=r^TWr}. $$

Substitute the reprojection residual:

$$ r_{ij}=p_{ij}^{*}-p_{ij}. $$

Then

$$ \begin{aligned}
\|p_{ij}^{*}-p_{ij}\|_{\Sigma_{ij}^{\mathrm{uncer}}}^{2}
&=(p_{ij}^{*}-p_{ij})^T\\
&\quad\Sigma_{ij}^{\mathrm{uncer}}\\
&\quad(p_{ij}^{*}-p_{ij}).
\end{aligned} $$

Now substitute Equation (4):

$$ \begin{aligned}
\|r_{ij}\|_{\Sigma_{ij}^{\mathrm{uncer}}}^{2}
&=r_{ij}^T
\operatorname{diag}\left(\frac{w_{ij}}{u_i'}\right)
r_{ij}.
\end{aligned} $$

If the scalar weight applies equally to both pixel-coordinate components, this becomes

$$ \boxed{
\|r_{ij}\|_{\Sigma_{ij}^{\mathrm{uncer}}}^{2}
=
\frac{w_{ij}}{u_i'}
(r_x^2+r_y^2)
} $$

This is the key mathematical result: the ordinary squared reprojection error is multiplied by the effective confidence weight $w_{ij}/u_i'$.

A terminology note: in conventional statistics, a Mahalanobis distance is written using the inverse covariance, or precision matrix.

Since the displayed Equation (4) uses $w_{ij}/u_i'$ directly as the weighting term, the derivation above treats $\Sigma_{ij}^{\mathrm{uncer}}$ as the matrix used in the weighted norm.


The excerpt alone does not establish whether the paper's symbol $\Sigma$ denotes a covariance or a precision matrix in the conventional statistical sense.

# Numerical example: How uncertainty reduces a moving object's influence


Suppose two pixels have the following values. For simplicity, assume $w_{ij}=1$.
Static scene pixel
High confidence

Residual magnitude = 2 pixels; uncertainty = 1

$$ E_{\mathrm{static}}=\frac{1}{1}(2^2)=4 $$



Moving object pixel
Low confidence

Residual magnitude = 10 pixels; uncertainty = 100

$$ E_{\mathrm{moving}}=\frac{1}{100}(10^2)=1 $$



Without uncertainty weighting, the moving pixel contributes $100$, compared with $4$ for the static pixel.

With the illustrative uncertainty values above, the moving pixel contributes only $1$.

Notice that the large residual has not disappeared. Instead, the optimizer is instructed to trust it less.

These numbers illustrate the weighting mechanism; they are not measurements reported by the paper.

**Deriving the pose-depth objective**


The paper defines the uncertainty-aware energy as

$$ \boxed{
\hat E(G',d')
=
\sum_{(i,j)\in E}
\|p_{ij}^{*}-p_{ij}\|_{\Sigma_{ij}^{\mathrm{uncer}}}^{2}
}
\tag{5}
$$

Here, $G'$ and $d'$ denote the pose and depth estimates being optimized.

Let's derive this equation from the individual pixel errors.

**Step 8.1 — Define the residual for every correspondence**


For each connected frame pair $(i,j)$,

$$ r_{ij}(G',d')
=
p_{ij}^{*}-\Pi(G_i',G_j',d_i',p_i). $$

The residual depends on the candidate poses and depth because changing either changes the geometric projection.

**Step 8.2 — Weight each residual**

Using Equation (4), the weighted squared error is


$$ E_{ij}
=
r_{ij}^{T}
\Sigma_{ij}^{\mathrm{uncer}}
r_{ij}. $$

Equivalently, under the scalar-weight interpretation,

$$ E_{ij}
=
\frac{w_{ij}}{u_i'}
\|r_{ij}\|^2. $$

**Step 8.3 — Aggregate all frame-pair errors**

The frame graph $E$ contains the frame pairs used for optimization. Summing their weighted errors gives


$$ \hat E(G',d')
=
\sum_{(i,j)\in E}
r_{ij}^{T}\Sigma_{ij}^{\mathrm{uncer}}r_{ij}. $$

Substituting the definition of $r_{ij}$ recovers Equation (5).

For a dense implementation, the sum also represents the relevant pixel correspondences within each frame pair, even when the pixel index is omitted from the compact notation.


**Step 8.4 — Minimize the energy**


The pose-depth update is conceptually

$$ \boxed{
(G^{\mathrm{new}},d^{\mathrm{new}})
=
\underset{G',d'}{\arg\min}\;
\hat E(G',d')
} $$

subject to the geometric parameterization of the poses and depths.

The result is the set of poses and depths that best explains the predicted correspondences while accounting for pixel uncertainty.

Importantly, during this step, the uncertainty weights are treated as fixed. This is what allows the paper to separate the optimization into manageable stages.

# How Gauss–Newton optimizes pose and depth


Equation (5) is nonlinear because the projected correspondence depends nonlinearly on camera pose and inverse depth.

Gauss–Newton solves such a problem by approximating the residual around the current estimate.

Let the current optimization state be $x$, representing the pose and depth variables. For a small update $\Delta x$, linearize the residual:

$$ r(x+\Delta x)
\approx r(x)+J\Delta x, $$

where

$$ J=\frac{\partial r}{\partial x} $$

is the Jacobian matrix.

For a weighted least-squares objective,

$$ E(x)=r(x)^TWr(x), $$

the linearized objective becomes

$$ E(x+\Delta x)
\approx
(r+J\Delta x)^T W(r+J\Delta x). $$

Expanding,

$$ \begin{aligned}
E(x+\Delta x)
\approx{}&
r^TWr\\
&+2\Delta x^T J^TWr\\
&+\Delta x^T J^TWJ\Delta x.
\end{aligned} $$

Differentiating with respect to $\Delta x$ and setting the derivative to zero yields

$$ 2J^TWr+2J^TWJ\Delta x=0. $$

Therefore,

$$ \boxed{
J^TWJ\Delta x=-J^TWr
} $$

This is the Gauss–Newton normal equation.

For uncertainty-aware BA, $W$ contains the uncertainty-dependent weights.

Consequently, high-uncertainty residuals contribute less to the normal equations and have less influence on the resulting pose-depth update.

The solver computes $\Delta x$, updates the state, and repeats as needed.

# Why the paper alternates between pose-depth and uncertainty optimization


The paper points out that jointly optimizing poses, depths, and uncertainties using Gauss–Newton is computationally expensive.


The reason is that all three sets of variables interact:
- Changing poses affects projected pixel locations.
- Changing depths affects the reconstructed 3D points and their projections.
- Changing uncertainties affects how strongly each residual contributes to the objective.


If all variables are optimized together, the solver must account for their coupled effects in one larger optimization problem.


Instead, the paper uses interleaved optimization.

1. Initialize pose, depth and uncertainty

2. Pose-depth refinement

    - Optimize \(G,d\) using Equation (5), holding uncertainty weights fixed.

3. Uncertainty optimization

    - Update the uncertainty estimates using the paper's uncertainty-optimization procedure.

4. Repeat the alternating updates


The supplied excerpt does not include the objective for Step 3, so we should not assume its exact loss function or derive its gradients without the next part of the paper.

# Section 3.3 — Uncertainty Optimization: Full step-by-step mathematical derivation

How does the system learn which pixels should have high uncertainty and which should have low uncertainty?

The key idea is to use DINOv2 visual-feature consistency across multiple views, instead of relying on reprojection error alone.




**1. Why do we need a separate uncertainty optimization?**


Imagine a camera observing a room containing a wall, a chair, and a moving person.


Static wall or chair

Their appearance is generally consistent across frames after accounting for camera motion. Their features should match well across views, so they should receive relatively low uncertainty.




Moving person

A rigid-motion projection may place a pixel at the wrong location because the person moves independently of the camera. Corresponding visual features can become inconsistent, indicating that the pixel should receive higher uncertainty.



Why not simply use the reprojection residual from Equation (5)?

Because a moving object may move so far that its predicted correspondence is unreliable.


The reprojection error then becomes a poor signal for deciding how uncertain the pixel should be.

Instead, Section 3.3 measures whether the visual features agree across frames.

The optimization has three main ingredients:

1. A feature-similarity loss that encourages inconsistent correspondences to receive high uncertainty.
2. A logarithmic prior that prevents uncertainty from growing without bound.
3. A learned mapping from DINOv2 features to uncertainty, which regularizes the uncertainty spatially.

**2. Step 1 — Extract visual features from two frames**


Consider two frames $ I_i $ and $ I_j $.

A feature extractor, FiT3D, which is based on a refined DINOv2 model, produces feature maps:

$$F_i=\Phi(I_i), \qquad F_j=\Phi(I_j). $$

Here, $ \Phi $ represents the feature-extraction network.

At a pixel $ p_i $, the feature vector is

$$F_i(p_i)\in\mathbb R^C, $$

where $ C $ is the feature dimension.

These are not raw RGB values. They are learned visual representations that can capture meaningful image patterns and semantic information.

For example, two image patches containing the same wall texture may have similar features even when their raw pixel intensities differ slightly due to lighting.

# Step 2 — Find the corresponding pixel in the other frame


Using the rigid-motion projection from Section 3.2, project pixel $ p_i$ into frame $ j$:

$$ \boxed{p_{ij}=\Pi(G_i,G_j,d_i,p_i)} $$

This projection estimates where the same 3D scene point should appear if the scene follows the camera-induced rigid motion.

Now sample the feature map of frame $ j$ at $ p_{ij}$:

$$ \boxed{F_{ij}=F_j(p_{ij})} $$

Since $ p_{ij}$ can fall between discrete pixel locations, the feature is obtained through bilinear interpolation.

For example, if the projected point lies between four neighboring pixels, bilinear interpolation combines their feature vectors using distance-dependent weights.

The uncertainty map is sampled at the same location:

$$ \boxed{u_{ij}'=u_j'(p_{ij})} $$

The two uncertainty values are therefore:
- $ u_i'$: uncertainty at the source pixel in frame $ i$.
- $ u_{ij}'$: uncertainty at its projected location in frame $ j$.
The method evaluates both directions across the image pair, allowing each frame to have its own uncertainty estimate.

# Step 3 — Derive the cosine similarity


The paper compares the feature vector in frame $i$ with the projected feature vector in frame $j$.

The cosine similarity is

$$ \boxed{
s_{ij}
=
\frac{F_i\cdot F_{ij}}
{\|F_i\|_2\|F_{ij}\|_2}
} $$

Let's derive each term.

**4.1 Dot product**

Suppose the feature vectors have $C$ components:

$$ F_i=(f_1,f_2,\ldots,f_C) $$


$$ F_{ij}=(g_1,g_2,\ldots,g_C). $$

Their dot product is

$$ F_i\cdot F_{ij}
=
\sum_{k=1}^{C}f_kg_k. $$

This measures how aligned the two feature vectors are.



**4.2 L2 norms**


The magnitude of the first feature vector is

$$ \|F_i\|_2=\sqrt{\sum_{k=1}^{C}f_k^2}. $$

Similarly,

$$ \|F_{ij}\|_2=\sqrt{\sum_{k=1}^{C}g_k^2}. $$

Dividing the dot product by both magnitudes normalizes the comparison, making it primarily dependent on feature direction rather than feature magnitude.


**4.3 Interpret the cosine similarity**



$$ s_{ij}=
\begin{cases}
1 & \text{same feature direction},\\
0 & \text{orthogonal feature vectors},\\
-1 & \text{opposite feature directions}.
\end{cases} $$

For example, consider two normalized feature vectors:

$$ F_i=(1,0), \qquad F_{ij}=(0.8,0.6). $$

Their magnitudes are both 1, so

$$ s_{ij}
=\frac{(1)(0.8)+(0)(0.6)}{1\cdot1}
=0.8. $$

The features are fairly similar.

If instead $F_{ij}=(0,1)$, then

$$ s_{ij}=\frac{0}{1\cdot1}=0. $$

The features are orthogonal, indicating low feature similarity.

Important: low feature similarity is a cue for multi-view inconsistency, not proof that an object is moving. Occlusion, lighting, viewpoint changes, and imperfect features can also produce inconsistency.

**Derive the similarity loss**


The paper defines

$$ \boxed{
E_{\mathrm{sim}}(u')
=
\sum_{(i,j)\in E}
\frac{1-s_{ij}}{u_i'u_{ij}'}
}
\tag{6} $$

where

$$ s_{ij}
=
\frac{F_i\cdot F_{ij}}
{\|F_i\|_2\|F_{ij}\|_2}. $$

Substituting cosine similarity into Equation (6),

$$ E_{\mathrm{sim}}(u')
=
\sum_{(i,j)\in E}
\frac{
1-\dfrac{F_i\cdot F_{ij}}
{\|F_i\|_2\|F_{ij}\|_2}
}{
u_i'u_{ij}'
}. $$

Let's understand this equation in three steps.

# Define the feature inconsistency


Let

$$ a_{ij}=1-s_{ij}. $$

This is the cosine distance used by the paper.

If the features match perfectly, \(s_{ij}=1\), so

$$ a_{ij}=1-1=0. $$

If the features are orthogonal, \(s_{ij}=0\), so

$$ a_{ij}=1-0=1. $$

If the cosine similarity is \(-1\), then

$$ a_{ij}=1-(-1)=2. $$

Thus, the smaller the cosine similarity, the larger the inconsistency term.

# Weight inconsistency using both uncertainties


For one correspondence, the loss is
$$
\boxed{
L_{ij}=\frac{a_{ij}}{u_i'u_{ij}'}
}
$$

Notice that the denominator contains the product of two uncertainties.
- $u_i'$ represents uncertainty in the source frame.
- $u_{ij}'$ represents uncertainty at the corresponding location in the target frame.
The loss therefore considers uncertainty at both ends of the correspondence.

**Understand the optimization pressure**


For a fixed positive inconsistency \(a_{ij}\), increasing either uncertainty reduces the similarity loss:

$$ \frac{\partial L_{ij}}{\partial u_i'}
=
-\frac{a_{ij}}{(u_i')^2u_{ij}'}, $$

and

$$ \frac{\partial L_{ij}}{\partial u_{ij}'}
=
-\frac{a_{ij}}{u_i'(u_{ij}')^2}. $$

When \(a_{ij}>0\), both derivatives are negative.

Therefore, minimizing \(E_{\mathrm{sim}}\) alone encourages the uncertainties to grow, especially where features are inconsistent.

However, this would allow a trivial solution: simply make all uncertainties extremely large so that every inconsistency contributes almost nothing to the loss.

**Numerical example — Feature inconsistency and uncertainty**


Suppose the existing correspondence weight is irrelevant to this loss and we examine one correspondence with the same uncertainty in both frames:


$$ u_i'=u_{ij}'=u.$$

Then
$$ L_{ij}=\frac{1-s_{ij}}{u^2}.$$

The table below compares two hypothetical correspondences.

$$\begin{array}{\|c\|c\|c\|} \hline \text{Feature similarity} & \text{Uncertainty } (u) & \text{Similarity loss} \\ \hline 0.9 & 1 & 0.1 \\ 0.1 & 1 & 0.9 \\ 0.1 & 3 & 0.1 \\ 0.9 & 3 & 0.0111 \\ \hline \end{array}$$

The loss is large when features disagree and uncertainty is low. Increasing uncertainty reduces that penalty.

This illustrates why the similarity loss must be combined with a prior that discourages excessive uncertainty.

# Equation (7) — Derive the logarithmic uncertainty prior


The paper introduces

$$ \boxed{
E_{\mathrm{prior}}(u')
=
\sum_i\log(u_i'+1)
}
\tag{7} $$

The purpose is to discourage the trivial solution $u_i'\rightarrow+\infty$.


# Step 7.1 — Why add a prior?

Recall the similarity loss:

$$ E_{\mathrm{sim}}
=
\sum_{(i,j)\in E}
\frac{a_{ij}}{u_i'u_{ij}'}. $$

Increasing uncertainty reduces this term.

Without an additional constraint, the optimizer could keep increasing uncertainty to make the loss approach zero.


The prior introduces a cost for large uncertainty values.


# Step 7.2 — Why use the logarithm?


For positive uncertainty, the prior for one pixel is

$$ P(u)=\log(u+1). $$

Its derivative is

$$ \boxed{
\frac{dP}{du}=\frac{1}{u+1}
} $$

Its second derivative is

$$ \frac{d^2P}{du^2}
=
-\frac{1}{(u+1)^2}. $$

Thus, the logarithmic prior increases with uncertainty but its slope decreases as uncertainty grows.


For example:

$$ P(0)=\log(1)=0, $$


$$ P(1)=\log(2)\approx0.693, $$


$$ P(3)=\log(4)\approx1.386. $$

The prior keeps penalizing increasing uncertainty, although the extra penalty per unit uncertainty becomes smaller at large values.


# Step 7.3 — Why add $1.0$?


The expression is

$$ \log(u_i'+1.0). $$

Because the Softplus output is positive, $u_i'\geq0$. Therefore,

$$ u_i'+1\geq1 $$

and consequently

$$ \boxed{\log(u_i'+1)\geq0}. $$

The bias term ensures that the logarithm is defined and nonnegative over the allowed uncertainty range.

Strictly speaking, positivity of the uncertainty already makes $\log(u_i')$ defined for $u_i'>0$; the $+1$ specifically shifts the prior and avoids negative prior values when $0<u_i'<1$.

# Equation (8) — Combine the two objectives


The total uncertainty cost is

$$ \boxed{
E_{\mathrm{uncer}}(u')
=
E_{\mathrm{sim}}(u')
+
\gamma_{\mathrm{prior}}E_{\mathrm{prior}}(u')
}
\tag{8} $$

Substituting Equations (6) and (7),

$$ \boxed{
E_{\mathrm{uncer}}(u')
=
\sum_{(i,j)\in E}
\frac{1-s_{ij}}{u_i'u_{ij}'}
+
\gamma_{\mathrm{prior}}
\sum_i\log(u_i'+1)
} $$

Here, \(\gamma_{\mathrm{prior}}\) controls the strength of the prior relative to the similarity loss.
- If \(\gamma_{\mathrm{prior}}\) is too small, uncertainty can grow excessively.
- If it is too large, uncertainty growth is strongly penalized, potentially preventing the model from expressing genuine dynamic inconsistency.
The optimization balances two competing objectives:

$$ \underbrace{\text{Feature inconsistency penalty}}_{\text{encourages uncertainty to increase}}
+
\underbrace{\text{Logarithmic prior}}_{\text{discourages excessive uncertainty}}. $$

For one correspondence, if we temporarily treat both uncertainties as the same variable \(u\), with \(a=1-s\), and assume the prior is applied once, the objective becomes

$$ L(u)=\frac{a}{u^2}+\gamma_{\mathrm{prior}}\log(u+1). $$

Its derivative is

$$ \frac{dL}{du}
=
-\frac{2a}{u^3}
+
\frac{\gamma_{\mathrm{prior}}}{u+1}. $$

A stationary point satisfies

$$ \boxed{
\frac{2a}{u^3}
=
\frac{\gamma_{\mathrm{prior}}}{u+1}
} $$

This equation illustrates the trade-off: larger inconsistency \(a\) creates stronger pressure to increase uncertainty, while the prior resists that increase. This is an illustrative simplified case; in the full system, the uncertainties are pixelwise and bidirectional, and each pixel may participate in many correspondences.

**Uncertainty regularization — Why learn uncertainty from DINOv2 features?**


So far, uncertainty has been treated as an independent variable for every pixel.

Suppose an image contains $H/8\times W/8$ uncertainty values.

Directly optimizing all of them independently can produce a noisy uncertainty map: neighboring pixels might receive very different uncertainty values even when they belong to the same object.

This is called spatial inconsistency or overfitting to local noise.

The paper instead learns a mapping from visual features to uncertainty:
$$
\boxed{u=\operatorname{Softplus}(\theta\cdot F)}
$$

Here:
- $F$ is the DINOv2 feature vector at a pixel.
- $\theta$ is a learnable parameter vector.
- $\theta\cdot F$ is a scalar affine score in the simplified notation used in the excerpt.
- Softplus transforms the score into a positive uncertainty.


If the affine layer has an explicit bias, its score can more generally be written $\theta^T F+b$; the paper's displayed formula omits a separate bias term.


**Step 9.1 — Why map features to uncertainty?**


Instead of independently learning an arbitrary uncertainty for every pixel, the system learns uncertainty from the features describing that pixel.

The resulting uncertainty is related to visual appearance and the learned representation. The shared parameters also constrain how uncertainty can vary across the feature map, helping reduce noisy, unconstrained pixelwise estimates.

This is the regularization mechanism described by the paper. It does not guarantee that every neighboring pixel receives identical uncertainty.


**Step 9.2 — Why use Softplus?**


The Softplus function is
$$
\boxed{\operatorname{Softplus}(z)=\log(1+e^z)}
$$

For a feature vector $F_i$,
$$
\boxed{u_i'=\log(1+e^{\theta^T F_i})}
$$

The output is positive:
$$
u_i'>0.
$$

That is important because uncertainty appears in the denominator of Equation (6), and the logarithmic prior assumes a valid nonnegative uncertainty.

Softplus is smooth and differentiable, making it suitable for gradient-based optimization.

**part A — Derive the Softplus derivative**


This is the key calculus step needed to understand how the learnable parameters are updated.

Start with
$$
u_i'=\log(1+e^{z_i}),
\qquad z_i=\theta^TF_i.
$$

Differentiate with respect to $z_i$:
$$
\frac{\partial u_i'}{\partial z_i}
=
\frac{1}{1+e^{z_i}}
\frac{\partial e^{z_i}}{\partial z_i}.
$$

Since
$$
\frac{\partial e^{z_i}}{\partial z_i}=e^{z_i},
$$

we get
$$
\frac{\partial u_i'}{\partial z_i}
=
\frac{e^{z_i}}{1+e^{z_i}}.
$$

Dividing the numerator and denominator by $e^{z_i}$,
$$
\frac{\partial u_i'}{\partial z_i}
=
\frac{1}{1+e^{-z_i}}.
$$

Therefore,
$$
\boxed{
\frac{\partial\,\operatorname{Softplus}(z_i)}
{\partial z_i}
=
\sigma(z_i)
=
\frac{1}{1+e^{-z_i}}
}
$$

where $\sigma$ is the sigmoid function.

The derivative of Softplus is sigmoid.

**part B — Derive the gradient with respect to $\theta$**

We now know that the uncertainty depends on $\theta$ through two operations:
$$
\theta
\longrightarrow
z_i=\theta^TF_i
\longrightarrow
u_i'=\operatorname{Softplus}(z_i).
$$

We want to compute
$$
\frac{\partial E_{\mathrm{uncer}}}{\partial\theta}.
$$

**Step 11.1 — Apply the chain rule**


Because the loss depends on $\theta$ through the uncertainty values,
$$
\frac{\partial E_{\mathrm{uncer}}}{\partial\theta}
=
\sum_i
\frac{\partial E_{\mathrm{uncer}}}{\partial u_i'}
\frac{\partial u_i'}{\partial\theta}.
$$

This is the multivariable chain rule: sum the contribution of every uncertainty output to the loss gradient.


**Step 11.2 — Differentiate the affine mapping**


Recall that
$$
z_i=\theta^TF_i.
$$

For a vector parameter $\theta$, the derivative of this scalar with respect to the vector is
$$
\boxed{
\frac{\partial z_i}{\partial\theta}=F_i
}
$$

assuming the usual column-vector convention.


**Step 11.3 — Apply the chain rule to Softplus**


Using
$$
\frac{\partial u_i'}{\partial z_i}
=
\frac{1}{1+e^{-z_i}},
$$

we obtain
$$
\frac{\partial u_i'}{\partial\theta}
=
\frac{\partial u_i'}{\partial z_i}
\frac{\partial z_i}{\partial\theta}.
$$

Hence,
$$
\boxed{
\frac{\partial u_i'}{\partial\theta}
=
\frac{1}{1+e^{-\theta^TF_i}}F_i
}
$$

This gives the sensitivity of the uncertainty output to the parameters.


**Step 11.4 — Substitute into the total gradient**


Substitute the previous result into the chain-rule expression:
$$
\frac{\partial E_{\mathrm{uncer}}}{\partial\theta}
=
\sum_i
\frac{\partial E_{\mathrm{uncer}}}{\partial u_i'}
\frac{1}{1+e^{-\theta^TF_i}}F_i.
$$

For parameter component $\theta_t$, this becomes
$$
\boxed{
g_t=
\sum_{i=0}^{N}
\frac{\partial E_{\mathrm{uncer}}}{\partial u_i'}
\frac{1}{1+e^{-\theta_{t-1}^{T}F_i}}
F_i
}
$$

This is the gradient expression shown in Equation (9), with the parameter vector written explicitly using a transpose.

The paper's notation uses $\theta_{t-1}\cdot F_i$ for the scalar feature score.

**Interpretation of the gradient:**
- $\frac{\partial E_{\mathrm{uncer}}}{\partial u_i'}$ tells us how changing a pixel's uncertainty affects the total loss.
- The sigmoid term tells us how sensitive the Softplus output is to its input.
- $F_i$ tells us how the input score changes with the learnable parameters.
- Summing over pixels combines these contributions into one parameter gradient.
This is the standard chain-rule structure of backpropagation.


# Derive the parameter update


The paper updates the learnable parameters using gradient descent with weight decay:
$$
\boxed{
\theta_t
=
\theta_{t-1}
-\lambda g_t
-\eta\theta_{t-1}
}
\tag{9}
$$

where:
- $\theta_{t-1}$: parameters before the update.
- $g_t$: gradient of the uncertainty loss with respect to the parameters.
- $\lambda$: gradient-descent learning rate.
- $\eta$: weight-decay coefficient.
- $\theta_t$: updated parameters.


Let's derive the update.


**Step 12.1 — Ordinary gradient descent**


For a differentiable objective $E(\theta)$, gradient descent updates parameters in the direction opposite to the gradient:
$$
\theta_t=\theta_{t-1}-\lambda g_t,
$$

where
$$
g_t=\frac{\partial E}{\partial\theta}.
$$

**Why subtract the gradient?**

The gradient points in the direction of the steepest local increase in the objective. Moving in the opposite direction tends to reduce the objective for a sufficiently small step.


**Step 12.2 — Add weight decay**


The paper additionally subtracts a term proportional to the current parameters:
$$
-\eta\theta_{t-1}.
$$

This discourages unnecessarily large parameter values and helps regularize the feature-to-uncertainty mapping.

Combining the two terms gives the update exactly as written in the paper:
$$
\theta_t
=
\theta_{t-1}
-\lambda g_t
-\eta\theta_{t-1}.
$$

Equivalently,
$$
\boxed{
\theta_t=(1-\eta)\theta_{t-1}-\lambda g_t
}
$$

One implementation detail is worth noting: this is the update as presented in your excerpt.

 In other weight-decay conventions, the decay term is multiplied by the learning rate, giving $\theta_t=\theta_{t-1}-\lambda(g_t+\eta\theta_{t-1})$. These conventions are not identical unless the coefficients are adjusted.

# Derive the uncertainty gradient itself


We have derived the gradient with respect to \$theta$, but what is

$$ \frac{\partial E_{\mathrm{uncer}}}{\partial u_i'}? $$

Let's derive it from Equation (8).

For a single correspondence, write

$$ L_{ij}=\frac{a_{ij}}{u_i'u_{ij}'},
\qquad a_{ij}=1-s_{ij}. $$

Treating the feature inconsistency as fixed during the uncertainty update,

$$ \frac{\partial L_{ij}}{\partial u_i'}
=
-\frac{a_{ij}}{(u_i')^2u_{ij}'}. $$

The prior for a pixel is

$$ P_i=\gamma_{\mathrm{prior}}\log(u_i'+1). $$

Its derivative is

$$ \frac{\partial P_i}{\partial u_i'}
=
\frac{\gamma_{\mathrm{prior}}}{u_i'+1}. $$

Therefore, for a source uncertainty that appears in this one correspondence and its prior, the total derivative is

$$ \boxed{
\frac{\partial L_{ij}+P_i}{\partial u_i'}
=
-\frac{1-s_{ij}}{(u_i')^2u_{ij}'}
+
\frac{\gamma_{\mathrm{prior}}}{u_i'+1}
} $$

In the complete system, each pixel may participate in multiple correspondences, and it can appear as either the source or target uncertainty.

Its total gradient accumulates all of those contributions, plus its prior derivative.

The two competing terms have opposite signs:
- The similarity term is nonpositive when $s_{ij}\leq1$, encouraging greater uncertainty when the features are inconsistent.

- The prior term is positive, discouraging uncertainty from growing indefinitely.
This balance is what makes the uncertainty optimization meaningful.

# Complete uncertainty optimization pipeline
1. Extract features
$$
F_i=\Phi(I_i),\qquad F_{ij}=F_j(p_{ij})
$$


2. Measure feature inconsistency
$$
a_{ij}=1-
\frac{F_i\cdot F_{ij}}
{\|F_i\|_2\|F_{ij}\|_2}
$$


3. Compute the uncertainty objective
$$
E_{\mathrm{uncer}}
=
\sum_{(i,j)}
\frac{a_{ij}}{u_i'u_{ij}'}
+
\gamma_{\mathrm{prior}}\sum_i\log(u_i'+1)
$$


4. Predict uncertainty from features
$$
u_i'=\operatorname{Softplus}(\theta^TF_i)
$$


5. Backpropagate through Softplus
$$
g_t=\sum_i
\frac{\partial E}{\partial u_i'}
\sigma(\theta_{t-1}^TF_i)F_i
$$


6. Update parameters
$$
\theta_t=(1-\eta)\theta_{t-1}-\lambda g_t
$$


# Section 3.4 — SLAM System

## 1. Keyframe initialization

The system initializes SLAM using 12 keyframes with sufficient camera motion. Keyframes provide useful multi-view constraints for estimating camera poses and scene geometry. Sufficient motion helps the system recover depth and camera movement from changes between images.

## 2. Metric depth regularization

DROID-SLAM initializes disparity to a constant value of 1, which may lead to inaccurate tracking in highly dynamic scenes. The authors use Metric3D to predict a depth map $D_i$ at resolution $H/8\times W/8$.

The depth-regularized bundle adjustment objective is

$$ \boxed{
E^+(G',d') = \underbrace{\sum_{(i,j)\in E} \left\|p_{ij}^* - p_{ij}\right\|_{\Sigma_{ij}^{\mathrm{uncer}}}^2}_{\text{Reprojection error}} + \underbrace{\gamma_d \sum_i \left\|d_i' - D_i\right\|^2}_{\text{Depth regularization}}
}
$$

- Reprojection term: Ensures that camera poses and estimated geometry explain image correspondences, while accounting for uncertainty.
- Depth regularization term: Encourages estimated depth to remain consistent with Metric3D's predictions.
- $\gamma_d$ controls the strength of depth regularization.

Note: Depth and disparity are different quantities ($d=1/D$), so the quantities being subtracted must be represented consistently in an actual implementation.

## 3. Frontend tracking — Local bundle adjustment

After initialization, new keyframes are processed incrementally using a sliding window.

For each new keyframe, the system optimizes camera poses, disparities, and dynamic uncertainties. It uses uncertainty-aware reprojection error and depth regularization to improve tracking while reducing the influence of moving objects.

The sliding window limits computation by focusing on a subset of relevant keyframes instead of optimizing the entire sequence for every new frame.

## 4. Backend refinement — Global bundle adjustment

After frontend tracking, the system performs global bundle adjustment over all keyframes to refine camera poses and disparities.

Unlike local BA, global BA uses constraints across the complete sequence. This can improve trajectory consistency and reduce accumulated pose errors.

During this stage, the dynamic-uncertainty parameters are frozen, but their existing uncertainty estimates can still influence the weighted reprojection error.

## 5. Why freeze uncertainty during global BA?

The uncertainty mapping

$$ u_i'=\operatorname{Softplus}(\theta^TF_i) $$

is designed to learn local uncertainty from image features within the sliding window.

The authors freeze its learnable parameters $\theta$ during global BA because this mapping is intended to regularize uncertainty locally, rather than across the entire trajectory.

Overall: Section 3.4 combines keyframe initialization, Metric3D depth regularization, local uncertainty-aware tracking, and global geometric refinement to improve SLAM accuracy in dynamic environments.

## 1. Metrics used

- ATE RMSE (Absolute Trajectory Error): Measures camera tracking accuracy by comparing estimated camera trajectories with ground-truth trajectories. Lower is better.
- Runtime (FPS): Measures processing speed as input frames divided by total processing time. Higher is better.
- 3D reconstruction quality: Qualitatively evaluated by comparing point clouds and Gaussian reconstructions for geometric accuracy and temporal consistency.

## 2. Main concerns

- Dynamic objects: Moving objects introduce unreliable reprojection errors and degrade camera pose estimation.
- Visually complex scenes: Motion blur, overexposure, clutter, and highly textured backgrounds can degrade mapping and uncertainty estimation.
- Initialization: Uncertainty estimation can be inaccurate when initial camera poses are unreliable.
- Runtime: The method is slightly slower than DROID-SLAM because of monocular depth estimation and DINOv2 feature extraction.
- Limitations of other methods: WildGS-SLAM can produce incorrect uncertainty estimates and reconstruction failures; feed-forward methods such as MonST3R and TTT3R show higher tracking errors in the reported experiments.

## 3. Advantages of DROID-W

- Better tracking accuracy: Achieves the best average ATE on the Bonn, TUM RGB-D, DyCheck, and DROID-W datasets among the reported comparisons.
- Better dynamic uncertainty estimation: Identifies dynamic regions and maintains stable confidence in static regions.
- Improved 3D reconstruction: Produces geometrically accurate and temporally consistent point clouds in challenging dynamic scenes.
- Outdoor robustness: Performs well on the proposed DROID-W outdoor dataset.
- Real-time performance: Achieves approximately 10 FPS and reports a 40× speedup over WildGS-SLAM.

## 4. Main ablation metric

The ablation study uses ATE RMSE on the Bonn RGB-D Dynamic dataset, in centimeters.

| Configuration                  | ATE RMSE ↓ |
| ------------------------------ | ---------- |
| Without uncertainty-aware BA   | 5.13       |
| Without monocular depth        | 3.30       |
| Without uncertainty decoupling | 2.57       |
| Without affine mapping         | 2.47       |
| Without weight decay           | 2.34       |
| Full DROID-W                   | 2.30       |

The full system achieves the lowest error, supporting the effectiveness of its individual components.

## 5. Overall conclusion

DROID-W's main advantage is robust camera tracking and 3D reconstruction in dynamic scenes through uncertainty-aware optimization. Its main concerns are unreliable uncertainty estimation during initialization and additional computation from depth estimation and feature extraction.